# Reproduce Identity2Vec (I2V) — 3-seed evaluation

Run top to bottom (**Shift+Enter**). You edit **two lines** in Step 1: `DATASET` and `SEEDS`.

The pipeline: pick a dataset (auto-aligned to a safe graph+labels version if needed) → train a fresh embedding **per seed** → score **node classification** (micro/macro/weighted F1) and **link prediction** (AUC) → report **mean ± std** across seeds.

All heavy logic lives in the project files (`make_labels.py`, `scripts/runner.py`); the notebook only sets knobs and calls them.


## Step 0 — Set up

Tells Python where the project is. Run once.


In [6]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working from:', Path.cwd())

Working from: /home/m-adam/identity2vec


## Step 1 — Choose dataset + seeds ⬅️ the only place you edit

`prepare_dataset` resolves the dataset to a **safe** version (e.g. author `citeseer` → aligned `citeseer_linqs`, built from one LINQS source so graph node-ids and labels match). It prints what it used and why, and never touches the original author files.


In [7]:
DATASET = "cora"        # cora | citeseer_linqs | pubmed | pubmed_linqs | amazon_computers | amazon_photo | coauthor_cs | coauthor_physics
SEEDS   = [42, 43, 44]      # 3 seeds for mean ± std
    
from make_labels import prepare_dataset
info = prepare_dataset(DATASET)   # -> {base, version, safe, edge_path, label_path}; builds aligned version if missing

Label file already exists: labels/cora.labels
dataset ready -> base=cora version=orig safe=cora
  edges  = input/cora.edgelist
  labels = labels/cora.labels


## Step 2 — Node classification (one embedding per seed)

For each seed: train a full-graph embedding → 70/30 stratified label split → logistic regression → **micro / macro / weighted F1**.
Embeddings are saved as `output/{base}_nc_{version}_s{seed}.emb`.


In [8]:
from runner import run_nodeclass_repeated
node_rows = run_nodeclass_repeated(info, seeds=SEEDS)

  [nc s42] reuse existing cora_nc_orig_s42.emb
  [nc s42] micro=0.7109 macro=0.6947 weighted=0.7091
  [nc s43] reuse existing cora_nc_orig_s43.emb
  [nc s43] micro=0.6790 macro=0.6655 weighted=0.6763
  [nc s44] reuse existing cora_nc_orig_s44.emb
  [nc s44] micro=0.6900 macro=0.6648 weighted=0.6866


## Step 3 — Link prediction (one split + embedding per seed)

For each seed: hide 30% of edges → train an embedding on the **70% train graph only** (leakage-free) → **AUC**.
Splits: `splits/{base}_lp_{version}_s{seed}_*`; embeddings: `output/{base}_lp_{version}_s{seed}.emb`.


In [9]:
from runner import run_linkpred_repeated
lp_rows = run_linkpred_repeated(info, seeds=SEEDS)

  [lp s42] reuse existing cora_lp_orig_s42.emb
  [lp s42] AUC=0.7980
  [lp s43] reuse existing cora_lp_orig_s43.emb
  [lp s43] AUC=0.8128
  [lp s44] reuse existing cora_lp_orig_s44.emb
  [lp s44] AUC=0.7923


## Step 4 — Results + summary (mean ± std)

Per-seed table, then the summary (`mean`, sample `std`, `delta = max − min`). Both saved to `results/`.


In [10]:
import os
from runner import summarize_seed_results

per_seed, summary = summarize_seed_results(node_rows, lp_rows)

tag = f"{info['base']}_{info['version']}"
res_dir = f"results/{info['safe']}"          # per-dataset subfolder
os.makedirs(res_dir, exist_ok=True)
per_seed.to_csv(f"{res_dir}/{tag}_per_seed.csv", index=False)
summary.to_csv(f"{res_dir}/{tag}_summary.csv", index=False)

def make_clean_table(per_seed, summary, task):
    task_rows = per_seed[per_seed["task"] == task].copy()

    if task == "nodeclass":
        metrics = ["micro_f1", "macro_f1", "weighted_f1"]
    elif task == "linkpred":
        metrics = ["auc"]
    else:
        raise ValueError(f"Unknown task: {task}")

    long_rows = task_rows.melt(
        id_vars=["dataset", "version", "task", "seed"],
        value_vars=metrics,
        var_name="metric",
        value_name="score",
    ).dropna(subset=["score"])

    seed_table = long_rows.pivot_table(
        index=["dataset", "version", "task", "metric"],
        columns="seed",
        values="score",
        aggfunc="first",
    ).reset_index()

    seed_table = seed_table.rename(columns={
        42: "seed_42",
        43: "seed_43",
        44: "seed_44",
    })

    summary_task = summary[summary["task"] == task].copy()

    clean = seed_table.merge(
        summary_task[["dataset", "version", "task", "metric", "mean", "std", "delta"]],
        on=["dataset", "version", "task", "metric"],
        how="left",
    )

    clean["final_score"] = clean.apply(
        lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}",
        axis=1,
    )

    # metric 1st column, task 2nd column on both tables; rows numbered from 1 not 0
    lead = ["metric", "task"]
    clean = clean[lead + [c for c in clean.columns if c not in lead]]
    clean = clean.sort_values("metric").reset_index(drop=True)
    clean.index = range(1, len(clean) + 1)

    return clean

node_table = make_clean_table(per_seed, summary, "nodeclass")
lp_table = make_clean_table(per_seed, summary, "linkpred")

node_table.to_csv(f"{res_dir}/{tag}_nodeclass_table.csv", index=False)
lp_table.to_csv(f"{res_dir}/{tag}_linkpred_table.csv", index=False)

print("\n=== Node Classification Table ===")
display(node_table)

print("\n=== Link Prediction Table ===")
display(lp_table)


=== Node Classification Table ===


,metric,task,dataset,version,seed_42,seed_43,seed_44,mean,std,delta,final_score
1,macro_f1,nodeclass,cora,orig,0.694725,0.665504,0.664802,0.675011,0.017077,0.029923,0.6750 ± 0.0171
2,micro_f1,nodeclass,cora,orig,0.710947,0.678967,0.690037,0.693317,0.016241,0.031980,0.6933 ± 0.0162
3,weighted_f1,nodeclass,cora,orig,0.709055,0.676254,0.686557,0.690622,0.016774,0.032801,0.6906 ± 0.0168



=== Link Prediction Table ===


,metric,task,dataset,version,seed_42,seed_43,seed_44,mean,std,delta,final_score
1,auc,linkpred,cora,orig,0.798032,0.812832,0.792344,0.801069,0.010576,0.020488,0.8011 ± 0.0106
